# 성남시 공공데이터 — 원본 CSV → Parquet 병합 과정

**목적**: `01_원본데이터/`의 월별 CSV 파일들을 `02_통합데이터/parquet/`의 Parquet 원본으로 보존  
**데이터 기간**: 2023.01 ~ 2025.12 (36개월, 일부 데이터는 2023.06~)  
**생성일**: 2026-04-08  

## 산출 파일 목록 (raw 원본 보존용)

| # | parquet 파일 | 소스 폴더 | 파일 수 | 비고 |
|---|---|---|---|---|
| 1 | `raw_card_merchant` | 카드/3. 가맹점 정보 | 36 | 원본 concat |
| 2 | `raw_sales_day` | 카드/10. 매출(day) | 36 | concat ⚠️ 558MB |
| 3 | `raw_corp_legal` | 기업/3. 법인 기업 | 36 | 원본 concat |
| 4 | `raw_corp_new` | 기업/4. 신규 기업 | 36 | 원본 concat |
| 5 | `raw_corp_move` | 기업/5. 이전 통계 | 35 | ⚠️ 2023-01 누락 |
| 6 | `raw_credit_inout` | 신용/3.전입+4.전출 | 36+36 | direction 구분 |
| 7 | `raw_credit_info` | 신용/8. 신용정보 | 36 | 원본 concat |
| 8 | `raw_credit_change` | 신용/6.전이+7.전이(경계) | 6+6 | ⚠️ 2023.06~ |
| 9 | `raw_work_stat` | 신용/9. 이동통계 | 6 | ⚠️ 2023.06~ |
| 10 | `raw_telecom_t4` | 통신/T4 | 36 | 원본 concat |

> **참고**: 집계 파일(merge_*, master_*)은 `multi-file-merge.ipynb`에서 생성합니다.

## 0. 환경 설정 및 공통 함수

In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import glob, os, re, warnings
warnings.filterwarnings("ignore")

# BASE 경로 설정 (노트북 위치 기준 상대 경로: notebooks/ → 03_EDA/ → 김재천/)
BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
ORIG = f"{BASE}/01_원본데이터"
PQ   = f"{BASE}/02_통합데이터/parquet"
os.makedirs(PQ, exist_ok=True)
print(f"BASE: {BASE}")
print(f"ORIG: {ORIG}")
print(f"PQ  : {PQ}")


In [ ]:
# ── 공통 상수 ────────────────────────────────────────────────
DISTRICT_MAP  = {41131: '수정구', 41133: '중원구', 41135: '분당구'}
SIGUNMAP      = {'성남시 분당구': 41135, '성남시 수정구': 41131, '성남시 중원구': 41133}

# ── 공통 함수 ────────────────────────────────────────────────
def load_csv(path):
    """CSV 로드: utf-8 / cp949 자동 시도, 파이프(|) 구분자 자동 감지
    (카드 매출 데이터는 2024.05 이후 파이프 구분자로 변경됨)
    """
    for enc in ['utf-8', 'cp949']:
        try:
            df = pd.read_csv(path, encoding=enc, nrows=1)
            if len(df.columns) == 1 and '|' in str(df.columns[0]):
                return pd.read_csv(path, sep='|', encoding=enc)
            return pd.read_csv(path, encoding=enc)
        except (UnicodeDecodeError, Exception):
            continue
    raise RuntimeError(f"파일 로드 실패: {path}")

def save_pq(df, name, skip_if_exists=False):
    """DataFrame → Parquet 저장. skip_if_exists=True 이면 이미 있을 경우 건너뜀"""
    path = f"{PQ}/{name}.parquet"
    if skip_if_exists and os.path.exists(path):
        size = os.path.getsize(path) / 1024**2
        print(f"⏭  {name}.parquet 이미 존재 ({size:.1f} MB) — 건너뜀")
        return
    pq.write_table(pa.Table.from_pandas(df), path, compression='snappy')
    size = os.path.getsize(path) / 1024**2
    print(f"💾 저장: {name}.parquet ({df.shape[0]:,}행 × {df.shape[1]}열 | {size:.1f} MB)")

def verify(df, name):
    """병합 결과 검증 출력"""
    print(f"{'─'*55}")
    print(f"✅ {name}: {df.shape[0]:,}행 × {df.shape[1]}열")
    if 'ym' in df.columns:
        print(f"   기간: {df['ym'].astype(str).min()} ~ {df['ym'].astype(str).max()}")
    elif 'stdr_ym' in df.columns:
        print(f"   기간: {df['stdr_ym'].min()} ~ {df['stdr_ym'].max()}")
    elif 'BS_YR_MON' in df.columns:
        print(f"   기간: {df['BS_YR_MON'].min()} ~ {df['BS_YR_MON'].max()}")
    null_pct = df.isnull().mean().mean() * 100
    print(f"   전체 결측률: {null_pct:.2f}%")
    print(f"   컬럼: {df.columns.tolist()}")

print("공통 함수 로드 완료")


## 1. 카드 데이터 (신한카드)

### 1-1. `raw_card_merchant.parquet` — 가맹점 원본

- **소스**: `카드/3. 가맹점 정보(대민)(mer_s)/` × 36개 월별 CSV  
- **변환**: 단순 concat (원본 컬럼 그대로 보존)  


In [ ]:
# raw_card_merchant: 원본 concat만 (컨럼 추가 없음)
files = sorted(glob.glob(f"{ORIG}/카드/3. 가맹점 정보(대민)(mer_s)/*.csv"))
print(f"파일 수: {len(files)}")

df_raw_mer = pd.concat([load_csv(f) for f in files], ignore_index=True)

verify(df_raw_mer, "raw_card_merchant")
display(df_raw_mer.head(3))
save_pq(df_raw_mer, "raw_card_merchant", skip_if_exists=True)


### 1-2. `raw_sales_day.parquet` — 매출 일별 원본

- **소스**: `카드/10. 매출(대민)(day)/` × 36개 월별 CSV  
- **변환**: 단순 concat (원본 보존)  
- **크기**: 약 **558 MB**, 87M+ 행 — ⚠️ 실행 시간 5~10분 소요  
- **주의**: `skip_if_exists=True`로 기존 파일 있으면 자동 건너뜀  
- **주요 컬럼**: `ta_ymd`(일자), `cty_rgn_no`(구), `card_tpbuz_nm_1`(업종), `amt`(매출액), `cnt`(건수)  


In [ ]:
# ⚠️ 대용량 파일 (558MB / 87M행) — 이미 생성된 경우 건너뜀
files = sorted(glob.glob(f"{ORIG}/카드/10. 매출(대민)(day)/*.csv"))
print(f"파일 수: {len(files)} | 예시: {os.path.basename(files[0])}")

out_path = f"{PQ}/raw_sales_day.parquet"
if os.path.exists(out_path):
    size = os.path.getsize(out_path) / 1024**2
    print(f"⏭  raw_sales_day.parquet 이미 존재 ({size:.0f} MB) — 건너뜀")
else:
    print("병합 시작 (시간이 걸립니다)...")
    chunks = []
    for i, f in enumerate(files):
        df_tmp = load_csv(f)
        chunks.append(df_tmp)
        if (i+1) % 6 == 0:
            print(f"  {i+1}/{len(files)} 완료...")
    df_sales = pd.concat(chunks, ignore_index=True)
    verify(df_sales, 'raw_sales_day')
    save_pq(df_sales, 'raw_sales_day')


## 2. 기업 데이터 (NPS 국민연금)

### 2-1. `raw_corp_legal.parquet` / `raw_corp_new.parquet` — 법인·신규기업 원본

- **소스**: `기업/3. 법인 기업(cnt)/` × 36 + `기업/4. 신규 기업(new)/` × 36  
- **변환**: 각각 단순 concat (원본 보존)  


In [ ]:
# raw_corp_legal: 법인 기업 원본 concat
files_corp = sorted(glob.glob(f"{ORIG}/기업/3. 법인 기업(cnt)/*.csv"))
df_raw_corp = pd.concat([load_csv(f) for f in files_corp], ignore_index=True)
verify(df_raw_corp, "raw_corp_legal")
display(df_raw_corp.head(3))
save_pq(df_raw_corp, "raw_corp_legal", skip_if_exists=True)

# raw_corp_new: 신규 기업 원본 concat
files_new = sorted(glob.glob(f"{ORIG}/기업/4. 신규 기업(new)/*.csv"))
df_raw_new = pd.concat([load_csv(f) for f in files_new], ignore_index=True)
verify(df_raw_new, "raw_corp_new")
display(df_raw_new.head(3))
save_pq(df_raw_new, "raw_corp_new", skip_if_exists=True)


### 2-2. `raw_corp_move.parquet` — 기업 이전 통계

- **소스**: `기업/5. 기업 이전 통계(nps_move_cnt)/` × **35개** 월별 CSV  
- ⚠️ **2023-01 데이터 원본 미제공** — 연간 집계 시 주의 필요  
- **변환**: concat + `in_gu_nm`(전입 구명) 파생  
- **주요 컬럼**: `stdr_ym`, `out_sigun_nm`(전출지), `in_sigun_nm`(전입지), `comp_cn`(이전 기업 수)  


In [ ]:
files = sorted(glob.glob(f"{ORIG}/기업/5. 기업 이전 통계(nps_move_cnt)/*.csv"))
print(f"파일 수: {len(files)} (⚠️ 202301 누락 — 202302 시작)")
print(f"범위: {os.path.basename(files[0])} ~ {os.path.basename(files[-1])}")

def extract_gu(sigun_nm):
    if pd.isna(sigun_nm): return None
    for gu in ['수정구', '중원구', '분당구']:
        if gu in str(sigun_nm): return gu
    return None

dfs = []
for f in files:
    df = load_csv(f)
    if len(df) == 0 or 'stdr_ym' not in df.columns:
        print(f"  SKIP: {os.path.basename(f)}")
        continue
    dfs.append(df)

df_move = pd.concat(dfs, ignore_index=True)
df_move['in_gu_nm'] = df_move['in_sigun_nm'].apply(extract_gu)

verify(df_move, 'raw_corp_move')
display(df_move.head(3))
save_pq(df_move, 'raw_corp_move', skip_if_exists=True)


## 3. 신용 데이터 (나이스평가정보)

### 3-1. `raw_credit_inout.parquet` — 전입·전출 원본

- **소스**: `신용/3. 전입(대민)(IN STAT)/` × 36 + `신용/4. 전출(대민)(OUT STAT)/` × 36  
- **변환**: 전입·전출 각각 concat 후 `direction` 컬럼('IN'/'OUT') 추가하여 수직 병합  
- **주요 컬럼**: `ym`, `BS_SIGNGU`(구 코드), `TOT_CNT`(이동 인원), `SELF_ENT`(자영업자 수), `YES_INCOME`(소득 있음), `AVG_INC`, `AVG_LOAN`  


In [ ]:
def load_inout(folder, direction):
    files = sorted(glob.glob(f"{ORIG}/신용/{folder}/*.csv"))
    print(f"  {direction}: {len(files)}개 파일")
    dfs = []
    for f in files:
        df = load_csv(f)
        dfs.append(df)
    df_all = pd.concat(dfs, ignore_index=True)
    df_all['direction'] = direction
    df_all['gu_nm'] = df_all['BS_SIGNGU'].map(DISTRICT_MAP)
    return df_all

df_in  = load_inout('3. 전입(대민)(IN STAT)',  'IN')
df_out = load_inout('4. 전출(대민)(OUT STAT)', 'OUT')

# ym 컬럼 파생 (전입은 BS_YR_MON, 전출도 동일)
for df in [df_in, df_out]:
    df['ym'] = df['BS_YR_MON'].astype(str)

# IN/OUT 구분 위한 dest 컬럼 (OUT은 전출지 기준)
df_in['dest_ym']     = np.nan
df_in['dest_sido']   = np.nan
df_in['dest_signgu'] = np.nan

df_io = pd.concat([df_in, df_out], ignore_index=True)

verify(df_io, 'raw_credit_inout')
display(df_io[['ym','BS_SIGNGU','gu_nm','direction','TOT_CNT','SELF_ENT','AVG_INC']].head(3))
save_pq(df_io, 'raw_credit_inout', skip_if_exists=True)


### 3-2. `raw_credit_info.parquet` — 신용정보 원본

- **소스**: `신용/8. 신용정보(대민)(DM_STAT)/` × 36개 월별 CSV  
- **변환**: 단순 concat (원본 보존)  


In [ ]:
# raw_credit_info: 신용정보 원본 concat
files = sorted(glob.glob(f"{ORIG}/신용/8. 신용정보(대민)(DM_STAT)/*.csv"))
print(f"파일 수: {len(files)}")

df_raw_credit = pd.concat([load_csv(f) for f in files], ignore_index=True)

verify(df_raw_credit, "raw_credit_info")
display(df_raw_credit.head(3))
save_pq(df_raw_credit, "raw_credit_info", skip_if_exists=True)


### 3-3. `raw_credit_change.parquet` — 신용등급 전이

- **소스**: `신용/6. 전이(CHANGE)/` × 6개 + `신용/7. 전이(경계)(CHAGE_L)/` × 6개  
- ⚠️ **2023.06~2025.12 (6개 스냅샷)** — 2023.06 이전 데이터 미제공  
- **변환**: `snap_ym`(기준 스냅샷 연월) 컬럼 파생 후 concat  
- **주요 컬럼**: `snap_ym`, `PRV_QNT`(이전 등급), `CUR_QNT`(현재 등급), `QNT_CNT`(해당 인원)  


In [ ]:
def load_change(folder, label):
    files = sorted(glob.glob(f"{ORIG}/신용/{folder}/*.csv"))
    print(f"  {label}: {len(files)}개 파일")
    dfs = []
    for f in files:
        df = load_csv(f)
        # 파일명에서 snap_ym 추출: GYEONGGI_CHANGE_2306.csv → 202306
        m = re.search(r'_(\d{4})\.csv', os.path.basename(f))
        if m:
            df['snap_ym'] = int('20' + m.group(1))
        df['source'] = label
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

df_chg   = load_change('6. 전이(CHANGE)',         'CHANGE')
df_chg_l = load_change('7. 전이(경계)(CHAGE_L)',  'CHANGE_L')

# 컬럼 통일 (CHANGE_L은 PRV_QNT_BND / CUR_QNT_BND 컬럼명 다름)
df_chg_l = df_chg_l.rename(columns={'PRV_QNT_BND': 'PRV_QNT', 'CUR_QNT_BND': 'CUR_QNT'})

df_change = pd.concat([df_chg, df_chg_l], ignore_index=True)
cols_front = ['snap_ym', 'source'] + [c for c in df_change.columns if c not in ['snap_ym','source']]
df_change = df_change[cols_front]

verify(df_change, 'raw_credit_change')
display(df_change.head(3))
save_pq(df_change, 'raw_credit_change', skip_if_exists=True)


### 3-4. `raw_work_stat.parquet` — 이동통계 (거주지×직장지)

- **소스**: `신용/9. 이동통계(대민)(WORK_STAT)/` × 6개 반기 CSV  
- ⚠️ **2023.06~2025.12 (6개 스냅샷)** — 2023.06 이전 미제공  
- **주요 컬럼**: `BS_SIGNGU`(기준 구), `H_BCD`(거주지 행정동), `W_BCD`(직장지 행정동), `AGE`(연령대), `TOT_CNT`(이동 인원)  
- **활용**: 성남시 내 통근 패턴, 구별 직주 근접도 분석  


In [ ]:
files = sorted(glob.glob(f"{ORIG}/신용/9. 이동통계(대민)(WORK_STAT)/*.csv"))
print(f"파일 수: {len(files)} (⚠️ 2306~2512 스냅샷만 존재)")

dfs = []
for f in files:
    df = load_csv(f)
    dfs.append(df)
    print(f"  {os.path.basename(f)}: {len(df):,}행")

df_work = pd.concat(dfs, ignore_index=True)
df_work['gu_nm'] = df_work['BS_SIGNGU'].map(DISTRICT_MAP)

verify(df_work, 'raw_work_stat')
display(df_work.head(3))
save_pq(df_work, 'raw_work_stat', skip_if_exists=True)


## 4. 통신 데이터 (SKT)

### 4-1. `raw_telecom_t4.parquet` — 전입 유동인구 원본 (T4)

- **소스**: `통신/T4/` × 36개 월별 CSV (`T4_GG_ADMI_INFLOW_YYYYMM_성남시.csv`)  
- **변환**: concat + `ym` 컬럼 추가  
- **주요 컬럼**: `ETL_YM`(기준 연월), `O_CTY_CD`(출발지), `D_CTY_CD`(도착지/성남시 구), `CNT`(이동 인원)  
- **의미**: 외부에서 성남시 각 구로 유입되는 인구 흐름  


In [ ]:
files = sorted(glob.glob(f"{ORIG}/통신/T4/*.csv"))
print(f"파일 수: {len(files)}")

dfs = []
for f in files:
    df = load_csv(f)
    dfs.append(df)

df_t4 = pd.concat(dfs, ignore_index=True)
df_t4['ym'] = df_t4['ETL_YM'].astype(str)

verify(df_t4, 'raw_telecom_t4')
display(df_t4.head(3))
save_pq(df_t4, 'raw_telecom_t4', skip_if_exists=True)


## 5. 병합 결과 요약

위 코드를 실행하면 `02_통합데이터/parquet/` 폴더에 10개의 raw parquet 파일이 생성됩니다.  
이 파일들은 원본 CSV의 보존용 아카이브이며, 분석용 집계 파일(merge_*, master_*)은  
`multi-file-merge.ipynb`에서 별도로 생성합니다.

In [ ]:
# 생성된 raw parquet 파일 확인
import os

raw_files = sorted([f for f in os.listdir(PQ) if f.startswith('raw_') and f.endswith('.parquet')])
print(f"raw parquet 파일 수: {len(raw_files)}\n")
for f in raw_files:
    size = os.path.getsize(f"{PQ}/{f}") / 1024**2
    print(f"  {f:<35s} {size:>8.1f} MB")